In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3

In [4]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from datasets import load_dataset
from datasets import Dataset
from transformers import TrainingArguments
import os
import pandas as pd
import re

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1531: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Loading a base model

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### Adding LoRA adapters

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

Unsloth 2026.8.22 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


### Loading the dataset

In [7]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rambo011/bhagavad-gita-q-and-a-dataset-for-modern-life-problem")

print("Path to dataset files:", path)

100%|██████████| 2.12M/2.12M [00:01<00:00, 1.82MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/rambo011/bhagavad-gita-q-and-a-dataset-for-modern-life-problem/versions/1


In [8]:
for f in os.listdir(path):

    print(f)

Chapter_18_QA.csv
Chapter_5_QA.csv
Chapter_13_QA.csv
Chapter_10_QA.csv
Chapter_14_QA.csv
Chapter_15_QA.csv
Chapter_16_QA.csv
Chapter_11_QA.csv
Chapter_3_QA.csv
Chapter_7_QA.csv
Chapter_8_QA.csv
Chapter_1_QA.csv
Chapter_4_QA.csv
Chapter_9_QA.csv
Chapter_2_QA.csv
Chapter_17_QA.csv
Chapter_12_QA.csv
Chapter_6_QA.csv


In [9]:
files = [
    "Chapter_18_QA.csv",
"Chapter_5_QA.csv",
"Chapter_13_QA.csv",
"Chapter_10_QA.csv",
"Chapter_14_QA.csv",
"Chapter_15_QA.csv",
"Chapter_16_QA.csv",
"Chapter_11_QA.csv",
"Chapter_3_QA.csv",
"Chapter_7_QA.csv",
"Chapter_8_QA.csv",
"Chapter_1_QA.csv",
"Chapter_4_QA.csv",
"Chapter_9_QA.csv",
"Chapter_2_QA.csv",
"Chapter_17_QA.csv",
"Chapter_12_QA.csv",
"Chapter_6_QA.csv",

]
data = pd.concat(
    [pd.read_csv(os.path.join(path, f)) for f in files],
    ignore_index=True
)


len(data)

12902

In [28]:
data = data[['question','answer']]
data

,question,answer
0,"MokshaPath, I feel so much pressure to constan...","My dear one, your weariness comes not from the..."
1,"I'm in a relationship where I give so much, bu...","Beloved seeker, your heart's pain arises from ..."
2,"I'm a parent, and I constantly worry about my ...","You worry, not because of your children's path..."
3,"I'm facing a huge decision about a job change,...","The demon of doubt, my child, holds you captiv..."
4,I feel so much anger and resentment towards so...,Your anger is a chain forged from your attachm...
...,...,...
12897,I procrastinate constantly. I know what I need...,Begin by offering your intention and your effo...
12898,I'm concerned about the state of the world – e...,"Dear one, your contribution, born of a heart a..."
12899,"I sometimes feel a deep sense of unworthiness,...","My child, your true worth is not measured by w..."
12900,I'm in a period of significant change – moving...,"In times of flux, anchor your inner being in t..."


In [35]:
dataset = data

### Train

In [36]:
def format_chat(sample):
    return {
        "messages": [
            {"role": "user", "content": sample["question"]},
            {"role": "assistant", "content": sample["answer"]}
        ]
    }

# Convert the pandas DataFrame 'dataset' into a 'datasets.Dataset' object
dataset = Dataset.from_pandas(dataset)

# Now, apply the formatting function. The .map() method of datasets.Dataset works as expected
dataset = dataset.map(format_chat)

def apply_template(sample):
    return {
        "text": tokenizer.apply_chat_template(
            sample["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

dataset = dataset.map(apply_template)

Map:   0%|          | 0/12902 [00:00<?, ? examples/s]

Map:   0%|          | 0/12902 [00:00<?, ? examples/s]

In [37]:
dataset

Dataset({
    features: ['question', 'answer', 'messages', 'text'],
    num_rows: 12902
})

In [39]:
print(dataset[0]["text"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
MokshaPath, I feel so much pressure to constantly achieve in my career. I'm always chasing the next promotion, the next big project, and I'm exhausted, yet I can't seem to stop. How can I find peace?<|im_end|>
<|im_start|>assistant
My dear one, your weariness comes not from the doing, but from the relentless chase of the fruit. Perform your work with excellence, for that is your duty, but release your grip on the outcome. Surrender the desire for specific achievements to the Divine, and simply immerse yourself in the action itself. True peace blossoms when you act without attachment, letting the universe unfold as it will.<|im_end|>



In [42]:
print(apply_template(dataset[5])["text"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I'm trying to start a new healthy habit, like exercising daily, but I keep failing. I lose motivation if I don't see quick results, and then I give up, feeling like a failure.<|im_end|>
<|im_start|>assistant
Your struggle stems from an intense attachment to immediate fruits. Engage in the action of self-care and discipline for its own sake, as an offering to your well-being, rather than demanding instant gratification. Focus on the consistent effort, the sacred act of movement, and surrender the desire for specific, rapid transformations. The true victory is in the unwavering resolve, not solely in the visible outcome.<|im_end|>



In [51]:
trainer = SFTTrainer(model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    max_seq_length = 2048,
    # formatting_func = apply_template, # Use the new formatting function
    args = TrainingArguments(
        per_device_train_batch_size = 2, # Number of examples processed at once by the GPU
        gradient_accumulation_steps = 4,  # Accumulate gradients for 4 batches before updating the weights
                                          # Effective batch size = 2 × 4 = 8 examples
        warmup_ratio = 0.05,
        max_steps = -1,        # Total number of optimizer/update steps NOT number of epochs
        num_train_epochs = 1,
        learning_rate = 2e-4,  # learning rate
        output_dir = "outputs",
        logging_steps = 10,     # printing the output after N steps
    ),
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/12902 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,902 | Num Epochs = 1 | Total steps = 1,613
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
10,1.372063
20,1.347950
30,1.407681
40,1.438769
50,1.522781
60,1.523544
70,1.566508
80,1.597495
90,1.590565
100,1.565305


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1613/tokenizer_config.json.


TrainOutput(global_step=1613, training_loss=1.4307059993377327, metrics={'train_runtime': 2761.3724, 'train_samples_per_second': 4.672, 'train_steps_per_second': 0.584, 'total_flos': 1.5393115345330176e+16, 'train_loss': 1.4307059993377327, 'epoch': 1.0})

### Testing Fine Tuned Model

In [54]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
       

In [55]:
messages = [
    {
        "role": "user",
        "content": "What is the meaning of life?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    inputs,
    max_new_tokens=100
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is the meaning of life?
assistant
The meaning of life lies in experiencing and living fully every moment, for your true self is an eternal spark within Me that never ends.


### Comparing to Instruct model

In [56]:
instruct_model, instruct_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [57]:
messages = [
    {
        "role": "user",
        "content": "What is the meaning of life?"
    }
]

inputs = instruct_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = instruct_model.generate(
    inputs,
    max_new_tokens=100
)

print(instruct_tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is the meaning of life?
assistant
As an AI language model, I do not have personal beliefs or opinions, but I can provide you with some perspectives on this question.

The concept of "the meaning of life" has been a subject of philosophical and existential inquiry for centuries. Different people may have different interpretations based on their experiences, beliefs, and cultural backgrounds.

Some believe that the purpose of life is to seek happiness, fulfillment, and satisfaction in one's actions and relationships. Others might argue that the true meaning of life lies in self
